## Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, BatchNormalization
from keras.regularizers import l1_l2
from keras.optimizers import Adam, RMSprop, SGD
from keras.callbacks import EarlyStopping

from imblearn.over_sampling import SMOTE
import optuna

## Load dataset

In [4]:
df = pd.read_csv("dataset.csv")
df

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,0.2350,...,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music
113996,113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,0.1170,...,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music
113997,113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,0.3290,...,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music
113998,113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,0.5060,...,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music


## Basic checks

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isnull().sum()

Unnamed: 0          0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          11

In [9]:
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Dropping unwanted/duplicate columns

In [10]:
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df.drop(columns=["Unnamed: 0", "track_id", "track_name", "album_name", "artists"], inplace=True, errors="ignore")

## Create target (binary classification — "Hit" song)

In [12]:
# Threshold-based target: popularity >= 70 → Hit (1), else Not Hit (0)
df["Is_Hit"] = (df["popularity"] >= 70).astype(int)
df["Is_Hit"].value_counts(normalize=True)

Is_Hit
0    0.952
1    0.048
Name: proportion, dtype: float64

## Feature engineering

In [13]:
df["duration_min"] = df["duration_ms"] / 60000
df["energy_valence_ratio"] = df["energy"] / (df["valence"] + 0.001)
df.drop(columns=["duration_ms", "popularity"], inplace=True)

## Train-test split

In [14]:
X = df.drop(columns=["Is_Hit"])
y = df["Is_Hit"]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=21, stratify=y_train_full)

## ColumnTransformer

In [15]:
num_cols = X.select_dtypes(exclude="object").columns
cat_cols = X.select_dtypes(include="object").columns

preprocessor = ColumnTransformer([
    ("Scaling", MinMaxScaler(), num_cols),
    ("Encoding", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
], remainder="drop")

X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

## Baseline ANN

In [16]:
model = Sequential([
    Input(shape=(X_train_t.shape[1],)),
    Dense(32, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="Adam", loss="binary_crossentropy", metrics=["accuracy"])

early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

model.fit(X_train_t, y_train,
          epochs=50,
          validation_data=(X_val_t, y_val),
          batch_size=256,
          callbacks=[early],
          verbose=2)

y_pred = np.where(model.predict(X_test_t) > 0.5, 1, 0)
accuracy_score(y_pred, y_test)

Epoch 1/50
285/285 - 6s - 23ms/step - accuracy: 0.9509 - loss: 0.1992 - val_accuracy: 0.9520 - val_loss: 0.1494
Epoch 2/50
285/285 - 2s - 7ms/step - accuracy: 0.9520 - loss: 0.1490 - val_accuracy: 0.9520 - val_loss: 0.1453
Epoch 3/50
285/285 - 2s - 6ms/step - accuracy: 0.9520 - loss: 0.1470 - val_accuracy: 0.9520 - val_loss: 0.1448
Epoch 4/50
285/285 - 2s - 8ms/step - accuracy: 0.9520 - loss: 0.1463 - val_accuracy: 0.9520 - val_loss: 0.1437
Epoch 5/50
285/285 - 3s - 9ms/step - accuracy: 0.9520 - loss: 0.1455 - val_accuracy: 0.9520 - val_loss: 0.1436
Epoch 6/50
285/285 - 3s - 9ms/step - accuracy: 0.9520 - loss: 0.1451 - val_accuracy: 0.9520 - val_loss: 0.1437
Epoch 7/50
285/285 - 2s - 8ms/step - accuracy: 0.9520 - loss: 0.1447 - val_accuracy: 0.9520 - val_loss: 0.1439
Epoch 8/50
285/285 - 3s - 11ms/step - accuracy: 0.9520 - loss: 0.1442 - val_accuracy: 0.9520 - val_loss: 0.1433
Epoch 9/50
285/285 - 3s - 10ms/step - accuracy: 0.9520 - loss: 0.1438 - val_accuracy: 0.9520 - val_loss: 0.143

0.9520175438596491

##  SMOTE 

In [17]:
smote = SMOTE()
X_train_res, y_train_res = smote.fit_resample(X_train_t, y_train)

## Optuna tuning

In [18]:
def objective(trial):
    lr_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    n_layers = trial.suggest_int('n_layers', 1, 4)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'RMSprop', 'SGD'])
    activation = trial.suggest_categorical('activation', ['tanh', 'relu'])
    batch_size1 = trial.suggest_categorical('batch_size', [32, 64, 128, 256, 512])

    model = Sequential()
    model.add(Input(shape=(X_train_res.shape[1],)))
    for i in range(n_layers):
        units = trial.suggest_int(f'units{i}', 8, 96)
        dropout = trial.suggest_float(f'dropout{i}', 0.0, 0.5)
        reg = trial.suggest_float(f'reg{i}', 1e-5, 1e-2, log=True)
        model.add(Dense(units, activation=activation, kernel_regularizer=l1_l2(l1=reg, l2=reg)))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))
    model.add(Dense(1, activation='sigmoid'))

    opt_map = {
        'Adam': Adam(learning_rate=lr_rate),
        'RMSprop': RMSprop(learning_rate=lr_rate),
        'SGD': SGD(learning_rate=lr_rate)
    }
    model.compile(optimizer=opt_map[optimizer_name], loss='binary_crossentropy',
                  metrics=[tf.keras.metrics.Recall(name='recall')])

    es = EarlyStopping(monitor='val_recall', mode='max', patience=5, restore_best_weights=True)
    history = model.fit(
        X_train_res, y_train_res,
        epochs=30, batch_size=batch_size1,
        validation_data=(X_val_t, y_val),
        callbacks=[es], verbose=0
    )
    val_recall = max(history.history['val_recall'])
    return val_recall

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=21))
study.optimize(objective, n_trials=15, show_progress_bar=True)

study.best_params

[I 2026-06-30 12:14:41,656] A new study created in memory with name: no-name-e1491148-05cc-4257-acab-e2e6dc2ba5d1
Best trial: 0. Best value: 0.996575:   7%|▋         | 1/15 [00:15<03:38, 15.61s/it]

[I 2026-06-30 12:14:57,294] Trial 0 finished with value: 0.9965753555297852 and parameters: {'learning_rate': 0.0001251554487151282, 'n_layers': 2, 'optimizer': 'Adam', 'activation': 'relu', 'batch_size': 512, 'units0': 19, 'dropout0': 0.08906233077974918, 'reg0': 0.00030746001885481724, 'units1': 84, 'dropout1': 0.37947191775605715, 'reg1': 0.008155589837765905}. Best is trial 0 with value: 0.9965753555297852.


Best trial: 0. Best value: 0.996575:  13%|█▎        | 2/15 [06:24<48:25, 223.47s/it]

[I 2026-06-30 12:21:06,226] Trial 1 finished with value: 0.9623287916183472 and parameters: {'learning_rate': 0.00330069279804087, 'n_layers': 2, 'optimizer': 'RMSprop', 'activation': 'relu', 'batch_size': 32, 'units0': 37, 'dropout0': 0.23007016689648802, 'reg0': 0.0004296403664910425, 'units1': 27, 'dropout1': 0.39993416346948824, 'reg1': 0.001513747182776292}. Best is trial 0 with value: 0.9965753555297852.


Best trial: 0. Best value: 0.996575:  20%|██        | 3/15 [07:55<32:36, 163.06s/it]

...

Best trial: 10. Best value: 0.998858:  93%|█████████▎| 14/15 [38:41<02:02, 122.49s/it]

[I 2026-06-30 12:53:23,563] Trial 13 finished with value: 0.9885844588279724 and parameters: {'learning_rate': 0.0009548711186217168, 'n_layers': 3, 'optimizer': 'Adam', 'activation': 'tanh', 'batch_size': 512, 'units0': 23, 'dropout0': 0.317459440846534, 'reg0': 0.00010313012869177684, 'units1': 82, 'dropout1': 0.22503467725632398, 'reg1': 0.0030490077192492967, 'units2': 54, 'dropout2': 0.004495636994420113, 'reg2': 0.0001678455453436085}. Best is trial 10 with value: 0.9988584518432617.


Best trial: 10. Best value: 0.998858: 100%|██████████| 15/15 [39:05<00:00, 156.37s/it]

[I 2026-06-30 12:53:47,288] Trial 14 finished with value: 0.9121004343032837 and parameters: {'learning_rate': 0.00010149440540116072, 'n_layers': 2, 'optimizer': 'Adam', 'activation': 'relu', 'batch_size': 512, 'units0': 51, 'dropout0': 0.11716236104310923, 'reg0': 1.1481441080614064e-05, 'units1': 84, 'dropout1': 0.4308225222638997, 'reg1': 0.000467655459373314}. Best is trial 10 with value: 0.9988584518432617.


{'learning_rate': 0.0009305715877131299,
 'n_layers': 3,
 'optimizer': 'Adam',
 'activation': 'tanh',
 'batch_size': 512,
 'units0': 11,
 'dropout0': 0.33469192330107084,
 'reg0': 3.232183326365325e-05,
 'units1': 94,
 'dropout1': 0.27164091119611844,
 'reg1': 0.009004727441270894,
 'units2': 16,
 'dropout2': 0.006794920242967045,
 'reg2': 1.1415983269620696e-05}

In [20]:
best = study.best_params
print(best)

{'learning_rate': 0.0009305715877131299, 'n_layers': 3, 'optimizer': 'Adam', 'activation': 'tanh', 'batch_size': 512, 'units0': 11, 'dropout0': 0.33469192330107084, 'reg0': 3.232183326365325e-05, 'units1': 94, 'dropout1': 0.27164091119611844, 'reg1': 0.009004727441270894, 'units2': 16, 'dropout2': 0.006794920242967045, 'reg2': 1.1415983269620696e-05}


## Final model with best Optuna params

In [22]:
model = Sequential()
model.add(Input(shape=(X_train_res.shape[1],)))

# Layer 1
model.add(Dense(11, activation='tanh', kernel_initializer='he_normal',
                 kernel_regularizer=l1_l2(l1=3.232183326365325e-05, l2=3.232183326365325e-05)))
model.add(BatchNormalization())
model.add(Dropout(0.33469192330107084))

# Layer 2
model.add(Dense(94, activation='tanh', kernel_initializer='he_normal',
                 kernel_regularizer=l1_l2(l1=0.009004727441270894, l2=0.009004727441270894)))
model.add(BatchNormalization())
model.add(Dropout(0.27164091119611844))

# Layer 3
model.add(Dense(16, activation='tanh', kernel_initializer='he_normal',
                 kernel_regularizer=l1_l2(l1=1.1415983269620696e-05, l2=1.1415983269620696e-05)))
model.add(BatchNormalization())
model.add(Dropout(0.006794920242967045))

model.add(Dense(1, activation='sigmoid', kernel_initializer="glorot_uniform"))

model.compile(optimizer=Adam(learning_rate=0.0009305715877131299), loss='binary_crossentropy',
              metrics=["accuracy", tf.keras.metrics.Recall(name='recall')])

es = EarlyStopping(monitor='val_recall', mode='max', patience=5, restore_best_weights=True)

history = model.fit(
    X_train_res, y_train_res,
    epochs=30, batch_size=512,
    validation_data=(X_val_t, y_val),
    callbacks=[es], verbose=0
)

## Evaluation

In [23]:
y_pred_test = np.where(model.predict(X_test_t) > 0.5, 1, 0)
print("Test Accuracy:", accuracy_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))

713/713 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step
Test Accuracy: 0.6364912280701754
              precision    recall  f1-score   support

           0       0.99      0.62      0.77     21706
           1       0.11      0.92      0.20      1094

    accuracy                           0.64     22800
   macro avg       0.55      0.77      0.48     22800
weighted avg       0.95      0.64      0.74     22800



## Save model + preprocessor

In [24]:
import pickle

with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

model.save('model.keras')
print("Saved preprocessor.pkl and model.keras")

Saved preprocessor.pkl and model.keras
